# 01 · Análise Exploratória (EDA)

Inspeção das 4 séries da B3 (PETR4, VALE3, AMER3, ITUB4): integridade, gaps, NaNs, dias sem pregão, splits/grupamentos e anomalias reais conhecidas.

A lógica vive em `src/`; este notebook só orquestra e visualiza. Dados lidos do cache `data/raw/` (offline) — ver `src/data.py` e `docs/adr/0007-coleta-e-tratamento-amer3.md`.

## Setup

No Colab, descomente o bloco de clone/instalação.

In [ ]:
# --- Colab ---
# !git clone https://github.com/Cerne17/NeuraTrade.git
# %cd NeuraTrade
# !pip install -e .

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CONFIG, set_seeds
from src import data

set_seeds()
TICKERS = CONFIG['tickers']
TRAIN_END = CONFIG['data']['train_end']
TICKERS

## Carregamento (cache offline)

Se o cache não existir, rode `data.cache_all()` **uma vez** (requer rede).

In [ ]:
try:
    series = data.load_all()
except FileNotFoundError:
    data.cache_all()
    series = data.load_all()

{t: df.shape for t, df in series.items()}

## Integridade das séries

Resumo por ticker (linhas, NaNs, volume zero, maiores quedas/altas).

In [ ]:
rep = pd.DataFrame({t: data.integrity_report(df) for t, df in series.items()}).T
rep

## Gaps e dias sem pregão

Diferença entre dias úteis (business days) do calendário e pregões efetivos revela feriados/halts. Não há feriados da B3 embutidos aqui; serve só de panorama.

In [ ]:
for t, df in series.items():
    full = pd.bdate_range(df.index.min(), df.index.max())
    faltam = len(full) - len(df.index)
    print(f'{t}: pregoes={len(df)}  business_days={len(full)}  diff(feriados/halts)={faltam}  NaNs={int(df.isna().sum().sum())}')

## Preço de fechamento ajustado (escala log)

`auto_adjust=True`: já corrige splits/dividendos. Linha tracejada = fim do treino.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, t in zip(axes.ravel(), TICKERS):
    s = series[t]['Close']
    ax.plot(s.index, s.values)
    ax.axvline(pd.Timestamp(TRAIN_END), color='red', ls='--', lw=1)
    ax.set_yscale('log')
    ax.set_title(t)
fig.suptitle('Close ajustado (log) — tracejado = fim do treino')
fig.tight_layout(); plt.show()

## Log-retornos diários

Feature principal do projeto. Picos = candidatos a anomalia.

In [ ]:
logret = {t: np.log(df['Close'] / df['Close'].shift(1)).dropna() for t, df in series.items()}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, t in zip(axes.ravel(), TICKERS):
    r = logret[t]
    ax.plot(r.index, r.values, lw=0.6)
    ax.axvline(pd.Timestamp(TRAIN_END), color='red', ls='--', lw=1)
    ax.set_title(f'{t}  (std={r.std():.4f})')
fig.suptitle('Log-retornos diários')
fig.tight_layout(); plt.show()

### Distribuição dos log-retornos

Caudas pesadas (leptocurtose) típicas de séries financeiras.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, t in zip(axes.ravel(), TICKERS):
    r = logret[t]
    ax.hist(r.values, bins=120)
    ax.set_title(f'{t}  curtose={r.kurtosis():.1f}  assimetria={r.skew():.2f}')
fig.suptitle('Distribuição dos log-retornos')
fig.tight_layout(); plt.show()

## Caso AMER3 (issue #7)

Série íntegra estruturalmente (0 NaN), mas com eventos extremos:
- **12/01/2023** — divulgação da fraude contábil; queda de ~78% no dia (log-ret ≈ -1.48). Anomalia **real** que o modelo deve detectar — não limpar.
- **14/11/2024** — salto de log-ret ≈ +1.03, compatível com **grupamento (reverse split)** não totalmente ajustado pelo yfinance — possível **artefato** a tratar.
- **Dias com volume zero** — halts/iliquidez.

Tratamento detalhado em `docs/adr/0007-coleta-e-tratamento-amer3.md`.

In [ ]:
amer = series['AMER3.SA']
print('Close em torno da fraude (jan/2023):')
print(amer.loc['2023-01-09':'2023-01-20', 'Close'])
print('\nMaiores |log-ret|:')
print(logret['AMER3.SA'].abs().nlargest(5))
print('\nDias com volume zero:', int((amer['Volume'] == 0).sum()))

In [ ]:
r = logret['AMER3.SA']
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(r.index, r.values, lw=0.7)
for d in ['2023-01-12', '2024-11-14']:
    ax.axvline(pd.Timestamp(d), color='red', ls=':', lw=1)
ax.set_title('AMER3 — log-retornos (vermelho: fraude 2023 e grupamento 2024)')
plt.show()

## Conclusões da EDA

- 4 séries com 3724 pregões (2010-01-04 → 2024-12-30), **sem NaN** após `auto_adjust`.
- Anomalias reais identificadas como âncoras narrativas (M6): PETR4 03/2020 (COVID), VALE3 01/2019 (Brumadinho), AMER3 01/2023 (fraude), ITUB4 10/2021.
- AMER3 exige atenção: artefato de grupamento em 11/2024 e dias de halt (ver ADR-0007).
- Log-retornos com caudas pesadas — coerente com o uso de threshold por percentil (ADR-0005).